In [29]:
import pandas as pd

df_claim = pd.read_csv('final-dataset(A1-31306samples)-train-topicmodel.csv')
df_claim.head()

,publication_number,country_code,kind_code,title,abstract,claims,publication_date,ipc_code,cpc_code,first_claim,claim_lengths,sub_classes,sub_class,abstract_lengths
0,US2020097067A1,US,A1,Artificial Intelligence System and Interactive...,"A reality interactive responding system, compr...","1 . A reality interactive responding system, c...",20200326,G06F9/448,G16H40/67,"1 . A reality interactive responding system, c...",200,"['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",123
1,US2020098473A1,US,A1,Data Storage and Retrieval System for Non-Cont...,A web-based interface enables medical personne...,What is claimed is: \n \n 1 . A da...,20200326,G16H40/67,H04L67/1097,What is claimed is: 1 . A data storage and ret...,200,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",142
2,US2020098451A1,US,A1,Hybrid analysis framework for prediction of ou...,A facility for predicting patient outcomes on ...,"1 . A method in a computing system, comprising...",20200326,G16H10/20,G16H10/20,"1 . A method in a computing system, comprising...",200,"['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",81
3,US2020098458A1,US,A1,Medical cannabis platform with physician and p...,"Through a physician&#39;s portal, a platform c...",What is claimed is: \n \n 1 . A me...,20200326,G16H80/00,A61K36/185,What is claimed is: 1 . A method for providing...,200,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",148
4,US2020093988A1,US,A1,Patient day planning systems and methods,"Infusion systems, infusion devices, and relate...",What is claimed is: \n \n 1 . A me...,20200326,G16H20/17,A61M2230/201,What is claimed is: 1 . A method of monitoring...,200,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",110


# preprocessing

In [30]:
def cleantext(df_claim): 
    
    df_claim['cleaned_Claim_text'] = df_claim['first_claim'].replace(r'\'|\"|\,|\.|\?|\+|\-|\/|\=|\(|\)|\n|"', '', regex=True)
    
    # convert to lowercase
    df_claim['cleaned_Claim_text'] = df_claim['cleaned_Claim_text'].str.lower()
    
    #remove numbers
    df_claim['cleaned_Claim_text'] =df_claim['cleaned_Claim_text'].replace(r'\d+', '', regex = True)
        
    #remove_symbols
    df_claim['cleaned_Claim_text']  = df_claim['cleaned_Claim_text'].replace(r'[^a-zA-Z0-9]', " ", regex=True)
    
    #remove punctuations 
    df_claim['cleaned_Claim_text'] = df_claim['cleaned_Claim_text'].replace(r'[[]!"#$%\'()\*+,-./:;<=>?^_`{|}]+',"", regex = True)
    
    #remove_URL(x):
    df_claim['cleaned_Claim_text']  = df_claim['cleaned_Claim_text'].replace(r'https.*$', "", regex = True)
    df_claim['cleaned_Claim_text'] = df_claim['cleaned_Claim_text'].replace("   ", " ", regex = True)
    df_claim['cleaned_Claim_text'] = df_claim['cleaned_Claim_text'].replace("  ", " ", regex = True)
   
    return df_claim

df_claim = cleantext(df_claim)
df_claim.head()

,publication_number,country_code,kind_code,title,abstract,claims,publication_date,ipc_code,cpc_code,first_claim,claim_lengths,sub_classes,sub_class,abstract_lengths,cleaned_Claim_text
0,US2020097067A1,US,A1,Artificial Intelligence System and Interactive...,"A reality interactive responding system, compr...","1 . A reality interactive responding system, c...",20200326,G06F9/448,G16H40/67,"1 . A reality interactive responding system, c...",200,"['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",123,a reality interactive responding system compr...
1,US2020098473A1,US,A1,Data Storage and Retrieval System for Non-Cont...,A web-based interface enables medical personne...,What is claimed is: \n \n 1 . A da...,20200326,G16H40/67,H04L67/1097,What is claimed is: 1 . A data storage and ret...,200,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",142,what is claimed is a data storage and retrieva...
2,US2020098451A1,US,A1,Hybrid analysis framework for prediction of ou...,A facility for predicting patient outcomes on ...,"1 . A method in a computing system, comprising...",20200326,G16H10/20,G16H10/20,"1 . A method in a computing system, comprising...",200,"['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",81,a method in a computing system comprising obt...
3,US2020098458A1,US,A1,Medical cannabis platform with physician and p...,"Through a physician&#39;s portal, a platform c...",What is claimed is: \n \n 1 . A me...,20200326,G16H80/00,A61K36/185,What is claimed is: 1 . A method for providing...,200,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",148,what is claimed is a method for providing netw...
4,US2020093988A1,US,A1,Patient day planning systems and methods,"Infusion systems, infusion devices, and relate...",What is claimed is: \n \n 1 . A me...,20200326,G16H20/17,A61M2230/201,What is claimed is: 1 . A method of monitoring...,200,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",110,what is claimed is a method of monitoring a ph...


In [31]:
import nltk
from nltk.corpus import stopwords

# Load the stop words
nltk.download('stopwords')
english_stop_words = set(stopwords.words('english'))

# Define a set of patent-specific stop words 337
patent_stop_words = set([
                        'a', 'an', 'the', 'is', 'are', 'was', 'were', 'will', 'shall', 'may', 'can', 'could',
                        'of', 'in', 'on', 'at', 'by', 'with', 'for', 'to', 'from', 'as', 'and', 'or', 'not',
                        'this', 'that', 'these', 'those', 'such', 'which', 'where', 'when', 'who', 'whom',
                        'what', 'how', 'why', 'be', 'have', 'has', 'having', 'had', 'do', 'does', 'doing',
                        'done', 'but', 'however', 'although', 'unless', 'until', 'since', 'while', 'before',
                        'after', 'during', 'through', 'over', 'under', 'above', 'below', 'between', 'among',
                        'within', 'without', 'against', 'via', 'via', 'due', 'onto', 'per', 'each', 'any',
                        'some', 'many', 'few', 'all', 'most', 'no', 'yes', 'ok', 'okk', 'nope', 'nopes', 'nay',
                        'more', 'less', 'least', 'other', 'another', 'somehow', 'thus', 'therefore',
                        'otherwise', 'instead', 'again', 'further', 'furthermore', 'also', 'besides', 'moreover',
                        'hence', 'thus', 'therefore', 'accordingly', 'consequently', 'so', 'then', 'thereby',
                        'otherwise', 'instead', 'again', 'further', 'furthermore', 'also', 'besides', 'moreover',
                        'hence', 'thus', 'therefore', 'accordingly', 'consequently', 'so', 'then', 'thereby',
                        'therein', 'thereupon', 'therewith', 'thereto', 'wherefore', 'forthwith', 'hereby',
                        'herein', 'hereupon', 'herewith', 'hereto', 'whereupon', 'aforesaid', 'heretofore',
                        'herewith', 'hereto', 'whereupon', 'aforesaid', 'heretofore', 'therefrom', 'thereof',
                        'therein', 'thereon', 'thereto', 'whereby', 'thereafter', 'hereafter', 'whenever',
                        'wherever', 'until', 'since', 'whenever', 'wherever', 'until', 'since', 'whereas', 'whilst',
                        'whereas', 'whilst', 'beside', 'beyond', 'during', 'about', 'around', 'above', 'below', 'before',
                        'after', 'without', 'within', 'between', 'among', 'against', 'under', 'over', 'through',
                        'into', 'onto', 'upon', 'am', 'are', 'is', 'was', 'were', 'be', 'been', 'being', 'have', 'has',
                        'had', 'having', 'do', 'does', 'did', 'doing', 'can', 'could', 'will', 'would', 'shall', 'should',
                        'may', 'might', 'must', 'ought', 'need', 'dare', 'used', 'use', 'using', 'uses', 'used', 'put',
                        'puts', 'putting', 'tell', 'tells', 'told', 'telling', 'ask', 'asks', 'asked', 'asking',
                        'make', 'makes', 'made', 'making', 'find', 'finds', 'found', 'finding', 'keep', 'keeps',
                        'kept', 'keeping', 'begin', 'begins', 'began', 'beginning', 'show', 'shows', 'showed',
                        'showing', 'say', 'says', 'said', 'saying', 'says', 'said', 'let', 'lets', 'letting', 'make',
                        'makes', 'made', 'making', 'put', 'puts', 'putting', 'seem', 'seems', 'seemed', 'seeming',
                        'need', 'needs', 'needed', 'needing', 'become', 'becomes', 'became', 'bec'])


# Merge the sets of stop words
stop_words = english_stop_words.union(patent_stop_words)

df_claim['cleaned_Claim_without_Stopwprd'] = df_claim['cleaned_Claim_text'].apply(lambda x: ' '.join([word for word in str(x).split() if word not in stop_words]))
df_claim.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,publication_number,country_code,kind_code,title,abstract,claims,publication_date,ipc_code,cpc_code,first_claim,claim_lengths,sub_classes,sub_class,abstract_lengths,cleaned_Claim_text,cleaned_Claim_without_Stopwprd
0,US2020097067A1,US,A1,Artificial Intelligence System and Interactive...,"A reality interactive responding system, compr...","1 . A reality interactive responding system, c...",20200326,G06F9/448,G16H40/67,"1 . A reality interactive responding system, c...",200,"['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",123,a reality interactive responding system compr...,reality interactive responding system comprisi...
1,US2020098473A1,US,A1,Data Storage and Retrieval System for Non-Cont...,A web-based interface enables medical personne...,What is claimed is: \n \n 1 . A da...,20200326,G16H40/67,H04L67/1097,What is claimed is: 1 . A data storage and ret...,200,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",142,what is claimed is a data storage and retrieva...,claimed data storage retrieval system nonconti...
2,US2020098451A1,US,A1,Hybrid analysis framework for prediction of ou...,A facility for predicting patient outcomes on ...,"1 . A method in a computing system, comprising...",20200326,G16H10/20,G16H10/20,"1 . A method in a computing system, comprising...",200,"['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",81,a method in a computing system comprising obt...,method computing system comprising obtaining i...
3,US2020098458A1,US,A1,Medical cannabis platform with physician and p...,"Through a physician&#39;s portal, a platform c...",What is claimed is: \n \n 1 . A me...,20200326,G16H80/00,A61K36/185,What is claimed is: 1 . A method for providing...,200,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",148,what is claimed is a method for providing netw...,claimed method providing networking communicat...
4,US2020093988A1,US,A1,Patient day planning systems and methods,"Infusion systems, infusion devices, and relate...",What is claimed is: \n \n 1 . A me...,20200326,G16H20/17,A61M2230/201,What is claimed is: 1 . A method of monitoring...,200,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",110,what is claimed is a method of monitoring a ph...,claimed method monitoring physiological condit...


In [32]:
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])
df_claim["Lemmatized-Claim"] = df_claim["cleaned_Claim_without_Stopwprd"].apply(lambda text: lemmatize_words(text))
df_claim.head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,publication_number,country_code,kind_code,title,abstract,claims,publication_date,ipc_code,cpc_code,first_claim,claim_lengths,sub_classes,sub_class,abstract_lengths,cleaned_Claim_text,cleaned_Claim_without_Stopwprd,Lemmatized-Claim
0,US2020097067A1,US,A1,Artificial Intelligence System and Interactive...,"A reality interactive responding system, compr...","1 . A reality interactive responding system, c...",20200326,G06F9/448,G16H40/67,"1 . A reality interactive responding system, c...",200,"['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",123,a reality interactive responding system compr...,reality interactive responding system comprisi...,reality interactive responding system comprisi...
1,US2020098473A1,US,A1,Data Storage and Retrieval System for Non-Cont...,A web-based interface enables medical personne...,What is claimed is: \n \n 1 . A da...,20200326,G16H40/67,H04L67/1097,What is claimed is: 1 . A data storage and ret...,200,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",142,what is claimed is a data storage and retrieva...,claimed data storage retrieval system nonconti...,claimed data storage retrieval system nonconti...
2,US2020098451A1,US,A1,Hybrid analysis framework for prediction of ou...,A facility for predicting patient outcomes on ...,"1 . A method in a computing system, comprising...",20200326,G16H10/20,G16H10/20,"1 . A method in a computing system, comprising...",200,"['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",81,a method in a computing system comprising obt...,method computing system comprising obtaining i...,method computing system comprising obtaining i...
3,US2020098458A1,US,A1,Medical cannabis platform with physician and p...,"Through a physician&#39;s portal, a platform c...",What is claimed is: \n \n 1 . A me...,20200326,G16H80/00,A61K36/185,What is claimed is: 1 . A method for providing...,200,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",148,what is claimed is a method for providing netw...,claimed method providing networking communicat...,claimed method providing networking communicat...
4,US2020093988A1,US,A1,Patient day planning systems and methods,"Infusion systems, infusion devices, and relate...",What is claimed is: \n \n 1 . A me...,20200326,G16H20/17,A61M2230/201,What is claimed is: 1 . A method of monitoring...,200,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",110,what is claimed is a method of monitoring a ph...,claimed method monitoring physiological condit...,claimed method monitoring physiological condit...


In [33]:
df_claim.head()

,publication_number,country_code,kind_code,title,abstract,claims,publication_date,ipc_code,cpc_code,first_claim,claim_lengths,sub_classes,sub_class,abstract_lengths,cleaned_Claim_text,cleaned_Claim_without_Stopwprd,Lemmatized-Claim
0,US2020097067A1,US,A1,Artificial Intelligence System and Interactive...,"A reality interactive responding system, compr...","1 . A reality interactive responding system, c...",20200326,G06F9/448,G16H40/67,"1 . A reality interactive responding system, c...",200,"['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",123,a reality interactive responding system compr...,reality interactive responding system comprisi...,reality interactive responding system comprisi...
1,US2020098473A1,US,A1,Data Storage and Retrieval System for Non-Cont...,A web-based interface enables medical personne...,What is claimed is: \n \n 1 . A da...,20200326,G16H40/67,H04L67/1097,What is claimed is: 1 . A data storage and ret...,200,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",142,what is claimed is a data storage and retrieva...,claimed data storage retrieval system nonconti...,claimed data storage retrieval system nonconti...
2,US2020098451A1,US,A1,Hybrid analysis framework for prediction of ou...,A facility for predicting patient outcomes on ...,"1 . A method in a computing system, comprising...",20200326,G16H10/20,G16H10/20,"1 . A method in a computing system, comprising...",200,"['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",81,a method in a computing system comprising obt...,method computing system comprising obtaining i...,method computing system comprising obtaining i...
3,US2020098458A1,US,A1,Medical cannabis platform with physician and p...,"Through a physician&#39;s portal, a platform c...",What is claimed is: \n \n 1 . A me...,20200326,G16H80/00,A61K36/185,What is claimed is: 1 . A method for providing...,200,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",148,what is claimed is a method for providing netw...,claimed method providing networking communicat...,claimed method providing networking communicat...
4,US2020093988A1,US,A1,Patient day planning systems and methods,"Infusion systems, infusion devices, and relate...",What is claimed is: \n \n 1 . A me...,20200326,G16H20/17,A61M2230/201,What is claimed is: 1 . A method of monitoring...,200,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",110,what is claimed is a method of monitoring a ph...,claimed method monitoring physiological condit...,claimed method monitoring physiological condit...


In [34]:
first_record = df_claim['Lemmatized-Claim'][1]
first_record

'claimed data storage retrieval system noncontiguous medical device data system comprising medical device connectable computer network subject occasional gap connectivity computer network configured automatically repeatedly capture status information medical device send message containing status information computer network network connectivity log configured automatically record time medical device connects computer network b time medical device disconnect computer network data store configured automatically store digital medium data medium file provide requested portion stored medium file response provision request wherein provision request includes index relative end medium file corresponds requested portion medium server connectable computer network configured automatically receive message computer network store status information received message data store wherein status information consecutive received message stored contiguously data store notwithstanding occasional gap connect

In [35]:
from gensim.models import Phrases
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel
from gensim.models.coherencemodel import CoherenceModel

# Tokenize the abstracts
docs = [doc.split() for doc in df_claim['Lemmatized-Claim']]

# Apply n-gram (1,3) using the Phrases class from Gensim
bigram_phrases = Phrases(docs, min_count=10)  # bigram model
trigram_phrases = Phrases(bigram_phrases[docs], min_count=10)  # trigram model

#quadgram_phraser = Phrases(trigram_phrases[bigram_phrases[docs]])
#docs = [quadgram_phraser[trigram_phrases[bigram_phrases[doc]]] for doc in docs]

docs = [trigram_phrases[bigram_phrases[doc]] for doc in docs]  # apply the trigram model to the tokenized documents

# Create the dictionary and document-term matrix using the Dictionary class from Gensim
id2word = Dictionary(docs)
id2word.filter_extremes(no_below=10, no_above=0.4)  # remove infrequent and very frequent words from the dictionary
corpus = [id2word.doc2bow(doc) for doc in docs]  # convert the tokenized documents to a bag-of-words corpus

# Train the LDA model using the LdaModel class from Gensim
num_topics = 30  # the number of topics to discover
lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, passes=10, random_state=42)

# Transform the corpus to get the topic distribution for each document
lda_output = lda_model[corpus]

# Calculate coherence metrics using the CoherenceModel class from Gensim
cm_cv = CoherenceModel(model=lda_model, corpus=corpus, texts=docs, dictionary=id2word, coherence='c_v')
coherence_cv = cm_cv.get_coherence()  # get the coherence score for the C_v metric

cm_npmi = CoherenceModel(model=lda_model, texts=docs, corpus=corpus, dictionary=id2word, coherence='c_npmi')
coherence_npmi = cm_npmi.get_coherence()  # get the coherence score for the c_npmi metric

cm_umass = CoherenceModel(model=lda_model, corpus=corpus, dictionary=id2word, coherence='u_mass')
coherence_umass = cm_umass.get_coherence()  # get the coherence score for the u_mass metric

# Print the coherence scores
print('C_v coherence:', coherence_cv)
print('c_npmi coherence:', coherence_npmi)
print('u_mass coherence:', coherence_umass)

C_v coherence: 0.38256263268167306
c_npmi coherence: -0.014029892174474494
u_mass coherence: -3.964854114233124


In [36]:
# Print the most significant words for each topic in the model
for topic_num in range(lda_model.num_topics):
    # Get the top 5 words for the current topic
    top_words = lda_model.show_topic(topic_num, topn=10)
    # Join the words into a single string for printing
    word_list = [word for word, _ in top_words]
    topic_words = " ".join(word_list)
    # Print the topic number and its top words
    print(f"Topic #{topic_num}: {topic_words}")

Topic #0: model location object characteristic representation virtual determining physical initial dynamic
Topic #1: medication type identifier prescription order identification associated member particular receiving
Topic #2: service digital file request service_provider facility criterion recited_claim voice station
Topic #3: user health activity input profile processor associated receiving determining device
Topic #4: healthcare item person product personal platform resource customer recipient card
Topic #5: point reference position motion body tissue movement area heart brain
Topic #6: mean medicine compliance pharmaceutical temperature stream center error assistance log
Topic #7: content component delivery adapted fluid layer clinical_trial phase key flow
Topic #8: monitoring physiological therapy status condition monitor plan alert change physiological_parameter
Topic #9: medical set feature training plurality condition candidate input comprises associated
Topic #10: sensor modul

In [37]:
# Get the most probable topic and its probability for each document
doc_topics = [sorted(lda_model.get_document_topics(doc), key=lambda x: x[1], reverse=True) for doc in lda_output]

In [38]:
# Create empty columns for topic number and probability
df_claim['topics'] = 0
df_claim['prob'] = 0.0

# Loop through each document and update the corresponding row in the dataframe   
for i, doc in enumerate(doc_topics):
    if len(doc) > 0:
        topic_num, topic_prob = doc[0]
        df_claim.at[i, 'topics'] = topic_num
        df_claim.at[i, 'prob'] = topic_prob  

In [39]:
df_claim

,publication_number,country_code,kind_code,title,abstract,claims,publication_date,ipc_code,cpc_code,first_claim,claim_lengths,sub_classes,sub_class,abstract_lengths,cleaned_Claim_text,cleaned_Claim_without_Stopwprd,Lemmatized-Claim,topics,prob
0,US2020097067A1,US,A1,Artificial Intelligence System and Interactive...,"A reality interactive responding system, compr...","1 . A reality interactive responding system, c...",20200326,G06F9/448,G16H40/67,"1 . A reality interactive responding system, c...",200,"['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",123,a reality interactive responding system compr...,reality interactive responding system comprisi...,reality interactive responding system comprisi...,26,0.294128
1,US2020098473A1,US,A1,Data Storage and Retrieval System for Non-Cont...,A web-based interface enables medical personne...,What is claimed is: \n \n 1 . A da...,20200326,G16H40/67,H04L67/1097,What is claimed is: 1 . A data storage and ret...,200,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",142,what is claimed is a data storage and retrieva...,claimed data storage retrieval system nonconti...,claimed data storage retrieval system nonconti...,26,0.462536
2,US2020098451A1,US,A1,Hybrid analysis framework for prediction of ou...,A facility for predicting patient outcomes on ...,"1 . A method in a computing system, comprising...",20200326,G16H10/20,G16H10/20,"1 . A method in a computing system, comprising...",200,"['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",81,a method in a computing system comprising obt...,method computing system comprising obtaining i...,method computing system comprising obtaining i...,24,0.219682
3,US2020098458A1,US,A1,Medical cannabis platform with physician and p...,"Through a physician&#39;s portal, a platform c...",What is claimed is: \n \n 1 . A me...,20200326,G16H80/00,A61K36/185,What is claimed is: 1 . A method for providing...,200,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",148,what is claimed is a method for providing netw...,claimed method providing networking communicat...,claimed method providing networking communicat...,26,0.298282
4,US2020093988A1,US,A1,Patient day planning systems and methods,"Infusion systems, infusion devices, and relate...",What is claimed is: \n \n 1 . A me...,20200326,G16H20/17,A61M2230/201,What is claimed is: 1 . A method of monitoring...,200,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",110,what is claimed is a method of monitoring a ph...,claimed method monitoring physiological condit...,claimed method monitoring physiological condit...,24,0.292083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31301,US2016253489A1,US,A1,User authentication system,A user authentication system performs user rec...,1 . A user authentication system comprising:\n...,20160901,G16H10/60,G06F21/32,1 . A user authentication system comprising: a...,200,"['G16H10/60', 'G06F21/32']","['G16H', 'G06F']",151,a user authentication system comprising a plu...,user authentication system comprising pluralit...,user authentication system comprising pluralit...,8,0.492714
31302,US2016253467A1,US,A1,"Diagnosis support apparatus and method, and no...",A diagnosis support apparatus for diagnosis of...,What is claimed is: \n \n 1 . A di...,20160901,G16H10/60,A61B5/743,What is claimed is: 1 . A diagnosis support ap...,200,"['G16H10/60', 'A61B5/743']","['G16H', 'A61B']",139,what is claimed is a diagnosis support apparat...,claimed diagnosis support apparatus diagnosis ...,claimed diagnosis support apparatus diagnosis ...,8,0.345636
31303,US2016253462A1,US,A1,Novel open-access scheduling system that optim...,A patient appointment schedule is generated fo...,1 . A medical appointment scheduling system co...,20160901,G16H40/20,G06F19/327,1 . A medical appointment scheduling system co...,200,"['G16H40/20', 'G06F19/327']","['G16H', 'G06F']",134,a medical appointment scheduling system compr...,medical appointment scheduling system comprisi...,medical appointment sch

In [40]:
df_Claim_topic=df_claim[["publication_number","title","first_claim", "Lemmatized-Claim","sub_classes","sub_class","topics","prob"]]
df_Claim_topic

,publication_number,title,first_claim,Lemmatized-Claim,sub_classes,sub_class,topics,prob
0,US2020097067A1,Artificial Intelligence System and Interactive...,"1 . A reality interactive responding system, c...",reality interactive responding system comprisi...,"['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",26,0.294128
1,US2020098473A1,Data Storage and Retrieval System for Non-Cont...,What is claimed is: 1 . A data storage and ret...,claimed data storage retrieval system nonconti...,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",26,0.462536
2,US2020098451A1,Hybrid analysis framework for prediction of ou...,"1 . A method in a computing system, comprising...",method computing system comprising obtaining i...,"['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",24,0.219682
3,US2020098458A1,Medical cannabis platform with physician and p...,What is claimed is: 1 . A method for providing...,claimed method providing networking communicat...,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",26,0.298282
4,US2020093988A1,Patient day planning systems and methods,What is claimed is: 1 . A method of monitoring...,claimed method monitoring physiological condit...,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",24,0.292083
...,...,...,...,...,...,...,...,...
31301,US2016253489A1,User authentication system,1 . A user authentication system comprising: a...,user authentication system comprising pluralit...,"['G16H10/60', 'G06F21/32']","['G16H', 'G06F']",8,0.492714
31302,US2016253467A1,"Diagnosis support apparatus and method, and no...",What is claimed is: 1 . A diagnosis support ap...,claimed diagnosis support apparatus diagnosis ...,"['G16H10/60', 'A61B5/743']","['G16H', 'A61B']",8,0.345636
31303,US2016253462A1,Novel open-access scheduling system that optim...,1 . A medical appointment scheduling system co...,medical appointment scheduling system comprisi...,"['G16H40/20', 'G06F19/327']","['G16H', 'G06F']",26,0.189817
31304,US2016249985A1,Interrelated point acquisition for navigated s...,"1 . A data processing, comprising a computer h...",data processing comprising computer processor ...,"['G16H20/40', 'G06F19/324']","['G16H', 'G06F']",10,0.457962


# prediction

In [41]:
import pandas as pd

df_Cliam_test = pd.read_csv('test-queries-USPTO(A1)-2023-G16H.csv')
df_Cliam_test.head()

,publication_numbers,abstract,first_claim,class_codes
0,US20230238130A1,A physiological sensor has light emitting sour...,1. A physiological monitoring device comprisin...,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/..."
1,US20230270344A1,A wearable monitoring device includes a band c...,"1. A monitoring device, comprising:\na band co...","A61B5/02405,A61B5/01,A61B5/16,A61B5/02055,A61B..."
2,US20230200909A1,A number of improvements are provided relating...,1-25. (canceled) 26. A method for guiding a fr...,"A61B17/17,A61B2090/061,A61B90/06,A61B2090/365,..."
3,US20230218347A1,Embodiments include a system for determining c...,1-184. (canceled) 185. A computer-implemented ...,"G06V10/46,G06V20/698,G06T2207/20112,G06T7/13,A..."
4,US20230063013A1,A community based response system for providin...,1. (canceled) 2. A community based response sy...,"H04M1/72418,G08B,G08,G08B25/016,G,H04W4/023,G1..."


# test1

In [42]:
df_Cliam_test.iloc[0]

publication_numbers                                      US20230238130A1
abstract               A physiological sensor has light emitting sour...
first_claim            1. A physiological monitoring device comprisin...
class_codes            A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...
Name: 0, dtype: object

# list of 10 queries

In [43]:
num_samples_to_predict = 10
num_of_topics = 5
results = []

# Assuming you have a list of 10 queries in test
queries = df_Cliam_test['first_claim'][:num_samples_to_predict]

for query in queries:
    query_tokens = trigram_phrases[bigram_phrases[query.split()]]  # Preprocess the query text
    query_bow = id2word.doc2bow(query_tokens)  # Convert to bag of words
    query_topic_dist = lda_model[query_bow]  # Get the topic distribution for the query
    top_topics = sorted(query_topic_dist, key=lambda x: x[1], reverse=True)[:num_of_topics]  # Get the top topics
    results.append(top_topics)

# Now, the 'results' list contains the top topics and their probabilities for each query
# You can access the results for a specific query like this:
for i, top_topics in enumerate(results):
    print(f"Query {i + 1}: Top {num_of_topics} Topics {top_topics}")



Query 1: Top 5 Topics [(27, 0.44127604), (21, 0.27422053), (19, 0.09982035), (7, 0.07621752), (8, 0.041562796)]
Query 2: Top 5 Topics [(27, 0.72168744), (5, 0.065727346), (22, 0.05070611), (10, 0.044271573), (26, 0.036214612)]
Query 3: Top 5 Topics [(5, 0.33110353), (25, 0.16941325), (0, 0.1385117), (14, 0.12607166), (13, 0.11857336)]
Query 4: Top 5 Topics [(7, 0.56655276), (19, 0.08871398), (18, 0.088426836), (16, 0.08838812), (26, 0.08824481)]
Query 5: Top 5 Topics [(22, 0.28644192), (0, 0.17371202), (8, 0.13085711), (24, 0.12460396), (26, 0.06498563)]
Query 6: Top 5 Topics [(10, 0.23528346), (24, 0.14771396), (22, 0.12574747), (11, 0.11810005), (16, 0.10117175)]
Query 7: Top 5 Topics [(10, 0.26552746), (1, 0.2566811), (22, 0.107471704), (25, 0.07724674), (17, 0.07318093)]
Query 8: Top 5 Topics [(3, 0.30850685), (10, 0.21200956), (5, 0.17465238), (11, 0.1551394), (22, 0.10522682)]
Query 9: Top 5 Topics [(25, 0.3387975), (14, 0.22255976), (3, 0.12774001), (10, 0.06323349), (23, 0.0485

In [44]:
# Create a new DataFrame to store the predicted topics
result_df_q = pd.DataFrame(columns=['query_publication_numbers', 'query_class_codes', 'query_claim', 'query_predicted_topics'])

# Populate the new DataFrame with the predicted topics
for i, top_topics in enumerate(results):
    query_publication_numbers = df_Cliam_test['publication_numbers'].iloc[i]
    query_class_codes = df_Cliam_test['class_codes'].iloc[i]
    query_claim = df_Cliam_test['first_claim'].iloc[i]
    predicted_topics = [topic_id for topic_id, _ in top_topics]
    
    # Append the data to the new DataFrame
    new_row = {'query_publication_numbers': query_publication_numbers,
               'query_class_codes': query_class_codes,
               'query_claim': query_claim,
               'query_predicted_topics': predicted_topics}
    
    result_df_q = pd.concat([result_df_q, pd.DataFrame([new_row])], ignore_index=True)

# Now, predicted_topics_df contains the desired structure of predicted topics for each query
result_df_q

,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics
0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,"[27, 21, 19, 7, 8]"
1,US20230270344A1,"A61B5/02405,A61B5/01,A61B5/16,A61B5/02055,A61B...","1. A monitoring device, comprising:\na band co...","[27, 5, 22, 10, 26]"
2,US20230200909A1,"A61B17/17,A61B2090/061,A61B90/06,A61B2090/365,...",1-25. (canceled) 26. A method for guiding a fr...,"[5, 25, 0, 14, 13]"
3,US20230218347A1,"G06V10/46,G06V20/698,G06T2207/20112,G06T7/13,A...",1-184. (canceled) 185. A computer-implemented ...,"[7, 19, 18, 16, 26]"
4,US20230063013A1,"H04M1/72418,G08B,G08,G08B25/016,G,H04W4/023,G1...",1. (canceled) 2. A community based response sy...,"[22, 0, 8, 24, 26]"
5,US20230095615A1,"A61,G,A61B5/14532,G16H40/00,H04L67/00,G16H40/6...",1. A method of monitoring analyte concentratio...,"[10, 24, 22, 11, 16]"
6,US20230005591A1,"A61,G09B19/003,G09B19/00,G,G16H40/00,G16H40/60...",1. A computer device configured to be in commu...,"[10, 1, 22, 25, 17]"
7,US20230001263A1,"A63B2225/105,A63B2225/09,A63B2230/00,G09B19/00...","1. An exercise machine, comprising:\na rail; a...","[3, 10, 5, 11, 22]"
8,US20230035015A1,"A61,A61B50/3001,A61B2050/0058,A61M5/3205,A61B5...",1-30. (canceled) 31. A method for preventing w...,"[25, 14, 3, 10, 23]"
9,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,"[11, 13, 9, 26, 18]"


In [45]:
import pandas as pd

# Create an empty DataFrame to store the results
result_df = pd.DataFrame(columns=['publication_number', 'title', 'first_claim', 'Lemmatized-Claim', 'sub_classes', 'sub_class', 'topics', 'prob', 'query_publication_numbers', 'query_class_codes', 'query_claim', 'query_predicted_topics'])

# Define the number of documents to retrieve for each topic
num_of_documents_to_retrieve = 10

for i, row in result_df_q.iterrows():
    query = row['query_claim']
    predicted_topics = row['query_predicted_topics']
    
    rows_to_append = []  # Create a list to store rows to be appended
    
    for topic_id in predicted_topics:
        # Filter 'df_Abstract_topic' to get the top 'num_of_documents_to_retrieve' documents for the current topic_id
        topic_documents = df_Claim_topic[df_Claim_topic['topics'] == topic_id]
        
        # Sort the documents by probability in descending order
        topic_documents = topic_documents.sort_values(by='prob', ascending=False).head(num_of_documents_to_retrieve)
        
        # Append the results to the 'rows_to_append' list
        for _, doc_row in topic_documents.iterrows():
            new_row = {
                'query_publication_numbers': row['query_publication_numbers'],
                'query_class_codes': row['query_class_codes'],
                'query_claim': query,
                'query_predicted_topics': [topic_id],  # Assign the current topic_id as a list
                'publication_number': doc_row['publication_number'],
                'title': doc_row['title'],
                'first_claim': doc_row['first_claim'],
                'Lemmatized-Claim': doc_row['Lemmatized-Claim'],
                'sub_classes': doc_row['sub_classes'],
                'sub_class': doc_row['sub_class'],
                'topics': doc_row['topics'],
                'prob': doc_row['prob']
            }
            rows_to_append.append(new_row)
    
    # Use pd.concat to append rows to result_df
    result_df = pd.concat([result_df, pd.DataFrame(rows_to_append)], ignore_index=True)

# Now, 'result_df' contains the top 10 most probable documents for each predicted topic list for each query
result_df

,publication_number,title,first_claim,Lemmatized-Claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics
0,US2022246286A1,Use of audio data for matching patients with h...,What is claimed is: 1 . A method comprising: o...,claimed method comprising obtaining computing ...,"['G16H50/20', 'G06N3/044']","['G16H', 'G06N']",27,0.479324,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27]
1,US2021050107A1,Medical treatment management,What is claimed is: 1 . A method comprising: r...,claimed method comprising receiving processor ...,"['G16H30/20', 'G16H20/00']","['G16H', 'G16H']",27,0.474824,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27]
2,US2022405475A1,System and method for improved feature definit...,1 - 28 . (canceled) 29 . A method of classifyi...,canceled method classifying speech transcript ...,"['G16H50/70', 'G16H50/20']","['G16H', 'G16H']",27,0.473163,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27]
3,US2002069092A1,Computerized method and system of assessing an...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q40/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467067,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27]
4,US2002062235A1,Computerized method and system for providing c...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q10/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467060,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27]
...,...,...,...,...,...,...,...,...,...,...,...,...
315,US2014180456A1,Wearable device assembly having athletic funct...,"What is claimed is: 1 . A method, comprising: ...",claimed method comprising receiving athletic a...,"['G16H20/30', 'Y10S482/901']","['G16H', 'Y10S']",26,0.512885,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26]
316,US2021272698A1,Personality based wellness coaching,What is claimed is: 1 . A computer-implemented...,claimed computerimplemented method comprising ...,"['G16H50/30', 'G16H40/67']","['G16H', 'G16H']",26,0.512585,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26]
317,US2018350452A1,Electronic medical record integration,1 . A method performed by a middleware applica...,method performed middleware application method...,"['G16H10/60', 'G16H10/60']","['G16H', 'G16H']",26,0.512565,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26]
318,US2007168220A1,Method and system for delivering educational c...,1 . A method for delivering a customized exerc...,method delivering customized exercise accompli...,"['G16H20/30', 'G06Q50/20']","['G16H', 'G06Q']",26,0.512392,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26]


In [46]:
# Add a new column to store the filtered codes
result_df['query_codes_G16H'] = ''

# Define a function to extract codes starting with 'G61H' from the class codes
def extract_G16H_codes(class_codes):
    codes = class_codes.split(',')
    return ','.join([code for code in codes if code.startswith('G16H')])

# Iterate through rows and update the 'query_codes_G61H' column
for index, row in result_df.iterrows():
    class_codes = row['query_class_codes']
    filtered_codes = extract_G16H_codes(class_codes)
    result_df.at[index, 'query_codes_G16H'] = filtered_codes

# Display the updated DataFrame
result_df

,publication_number,title,first_claim,Lemmatized-Claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics,query_codes_G16H
0,US2022246286A1,Use of audio data for matching patients with h...,What is claimed is: 1 . A method comprising: o...,claimed method comprising obtaining computing ...,"['G16H50/20', 'G06N3/044']","['G16H', 'G06N']",27,0.479324,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
1,US2021050107A1,Medical treatment management,What is claimed is: 1 . A method comprising: r...,claimed method comprising receiving processor ...,"['G16H30/20', 'G16H20/00']","['G16H', 'G16H']",27,0.474824,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
2,US2022405475A1,System and method for improved feature definit...,1 - 28 . (canceled) 29 . A method of classifyi...,canceled method classifying speech transcript ...,"['G16H50/70', 'G16H50/20']","['G16H', 'G16H']",27,0.473163,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
3,US2002069092A1,Computerized method and system of assessing an...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q40/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467067,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
4,US2002062235A1,Computerized method and system for providing c...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q10/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467060,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,US2014180456A1,Wearable device assembly having athletic funct...,"What is claimed is: 1 . A method, comprising: ...",claimed method comprising receiving athletic a...,"['G16H20/30', 'Y10S482/901']","['G16H', 'Y10S']",26,0.512885,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G..."
316,US2021272698A1,Personality based wellness coaching,What is claimed is: 1 . A computer-implemented...,claimed computerimplemented method comprising ...,"['G16H50/30', 'G16H40/67']","['G16H', 'G16H']",26,0.512585,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G..."
317,US2018350452A1,Electronic medical record integration,1 . A method performed by a middleware applica...,method performed middleware application method...,"['G16H10/60', 'G16H10/60']","['G16H', 'G16H']",26,0.512565,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G..."
318,US2007168220A1,Method and system for delivering educational c...,1 . A method for delivering a customized exerc...,method delivering customized exercise accompli...,"['G16H20/30', 'G06Q50/20']","['G16H', 'G06Q']",26,0.512392,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G..."


In [47]:
import ast


# Create a new column to store the common code
result_df['exact_match_code'] = ''

# Iterate through rows and compare 'sub_classes' and 'query_codes_G16H'
for index, row in result_df.iterrows():
    sub_classes_str = row['sub_classes']  # Data format in this field "['A61B5/00', 'G16H40/67']"
    query_codes_G16H = row['query_codes_G16H']  # Data format in this field 'G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H'
    
    # Convert the sub_classes string to a list
    sub_classes = ast.literal_eval(sub_classes_str)
    
    # Split the codes into lists
    sub_class_list = [code.strip() for code in sub_classes]
    query_codes_list = query_codes_G16H.split(',')
    
    # Check for common codes
    exact_match_code = [code for code in sub_class_list if code in query_codes_list]
    
    # Join the common codes into a single string
    exact_match_code_str = ','.join(exact_match_code)
    
    # Update the 'exact_match_code' column with the exact_match_code
    result_df.at[index, 'exact_match_code'] = exact_match_code_str
    
    # Debugging statements
    #print(f'Row {index}: sub_classes={sub_classes}, query_codes_G16H={query_codes_G16H}, common_codes={common_codes_str}')

# Display the updated DataFrame
result_df

,publication_number,title,first_claim,Lemmatized-Claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics,query_codes_G16H,exact_match_code
0,US2022246286A1,Use of audio data for matching patients with h...,What is claimed is: 1 . A method comprising: o...,claimed method comprising obtaining computing ...,"['G16H50/20', 'G06N3/044']","['G16H', 'G06N']",27,0.479324,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",
1,US2021050107A1,Medical treatment management,What is claimed is: 1 . A method comprising: r...,claimed method comprising receiving processor ...,"['G16H30/20', 'G16H20/00']","['G16H', 'G16H']",27,0.474824,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",
2,US2022405475A1,System and method for improved feature definit...,1 - 28 . (canceled) 29 . A method of classifyi...,canceled method classifying speech transcript ...,"['G16H50/70', 'G16H50/20']","['G16H', 'G16H']",27,0.473163,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",
3,US2002069092A1,Computerized method and system of assessing an...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q40/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467067,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",G16H40/67
4,US2002062235A1,Computerized method and system for providing c...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q10/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467060,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",G16H40/67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,US2014180456A1,Wearable device assembly having athletic funct...,"What is claimed is: 1 . A method, comprising: ...",claimed method comprising receiving athletic a...,"['G16H20/30', 'Y10S482/901']","['G16H', 'Y10S']",26,0.512885,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G...",
316,US2021272698A1,Personality based wellness coaching,What is claimed is: 1 . A computer-implemented...,claimed computerimplemented method comprising ...,"['G16H50/30', 'G16H40/67']","['G16H', 'G16H']",26,0.512585,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G...",
317,US2018350452A1,Electronic medical record integration,1 . A method performed by a middleware applica...,method performed middleware application method...,"['G16H10/60', 'G16H10/60']","['G16H', 'G16H']",26,0.512565,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G...",
318,US2007168220A1,Method and system for delivering educational c...,1 . A method for delivering a customized exerc...,method delivering customized exercise accompli...,"['G16H20/30', 'G06Q50/20']","['G16H', 'G06Q']",26,0.512392,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G...",


In [48]:
# Calculate the count of 'exact_match_code' for each group and assign it to all rows within the group
result_df['count_exact_match'] = result_df.groupby('query_publication_numbers')['exact_match_code'].transform(lambda x: x[x != ''].count())
result_df

,publication_number,title,first_claim,Lemmatized-Claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics,query_codes_G16H,exact_match_code,count_exact_match
0,US2022246286A1,Use of audio data for matching patients with h...,What is claimed is: 1 . A method comprising: o...,claimed method comprising obtaining computing ...,"['G16H50/20', 'G06N3/044']","['G16H', 'G06N']",27,0.479324,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
1,US2021050107A1,Medical treatment management,What is claimed is: 1 . A method comprising: r...,claimed method comprising receiving processor ...,"['G16H30/20', 'G16H20/00']","['G16H', 'G16H']",27,0.474824,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
2,US2022405475A1,System and method for improved feature definit...,1 - 28 . (canceled) 29 . A method of classifyi...,canceled method classifying speech transcript ...,"['G16H50/70', 'G16H50/20']","['G16H', 'G16H']",27,0.473163,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
3,US2002069092A1,Computerized method and system of assessing an...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q40/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467067,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",G16H40/67,6
4,US2002062235A1,Computerized method and system for providing c...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q10/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467060,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",G16H40/67,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,US2014180456A1,Wearable device assembly having athletic funct...,"What is claimed is: 1 . A method, comprising: ...",claimed method comprising receiving athletic a...,"['G16H20/30', 'Y10S482/901']","['G16H', 'Y10S']",26,0.512885,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G...",,0
316,US2021272698A1,Personality based wellness coaching,What is claimed is: 1 . A computer-implemented...,claimed computerimplemented method comprising ...,"['G16H50/30', 'G16H40/67']","['G16H', 'G16H']",26,0.512585,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G...",,0
317,US2018350452A1,Electronic medical record integration,1 . A method performed by a middleware applica...,method performed middleware application method...,"['G16H10/60', 'G16H10/60']","['G16H', 'G16H']",26,0.512565,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G...",,0
318,US2007168220A1,Method and system for delivering educational c...,1 . A method for delivering a customized exerc...,method delivering customized exercise accompli...,"['G16H20/30', 'G06Q50/20']","['G16H', 'G06Q']",26,0.512392,US20230218172A1,"G06N3/045,G06T2207/00,A61,G06N3/00,G06N3/04,G0...",1. A method for selectively presenting images ...,[26],"G16H30/40,G16H30/20,G16H,G16H50/20,G16H30/00,G...",,0


In [49]:
filtered_df = result_df[result_df['query_publication_numbers'] == 'US20230238130A1']
filtered_df

,publication_number,title,first_claim,Lemmatized-Claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics,query_codes_G16H,exact_match_code,count_exact_match
0,US2022246286A1,Use of audio data for matching patients with h...,What is claimed is: 1 . A method comprising: o...,claimed method comprising obtaining computing ...,"['G16H50/20', 'G06N3/044']","['G16H', 'G06N']",27,0.479324,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
1,US2021050107A1,Medical treatment management,What is claimed is: 1 . A method comprising: r...,claimed method comprising receiving processor ...,"['G16H30/20', 'G16H20/00']","['G16H', 'G16H']",27,0.474824,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
2,US2022405475A1,System and method for improved feature definit...,1 - 28 . (canceled) 29 . A method of classifyi...,canceled method classifying speech transcript ...,"['G16H50/70', 'G16H50/20']","['G16H', 'G16H']",27,0.473163,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
3,US2002069092A1,Computerized method and system of assessing an...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q40/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467067,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",G16H40/67,6
4,US2002062235A1,Computerized method and system for providing c...,What is claimed is: 1 . A method of estimating...,claimed method estimating liability accident c...,"['G06Q10/00', 'G16H40/67']","['G06Q', 'G16H']",27,0.467060,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",G16H40/67,6
5,US2021375430A1,"Methods, systems, and devices for generating a...",What is claimed is: 1 . A system for generatin...,claimed system generating refreshment instruct...,"['G16H50/20', 'G16H20/60']","['G16H', 'G16H']",27,0.463687,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
6,US2022028535A1,Method for mitigating disease transmission in ...,I claim: 1 . A method for mitigating disease t...,claim method mitigating disease transmission f...,"['G16H40/20', 'G06V20/52']","['G16H', 'G06V']",27,0.461608,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
7,US2020320391A1,Multilayer perceptron based network to identif...,What is claimed is: 1 . A method for training ...,claimed method training baseline risk model co...,"['G06N3/04', 'G16H50/70']","['G06N', 'G16H']",27,0.433526,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
8,US2019057760A1,Automated medical note generation system utili...,We claim: 1 . A computer-implemented system fo...,claim computerimplemented system preparing med...,"['G16H10/60', 'G10L15/26']","['G16H', 'G10L']",27,0.419944,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[27],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,6
9,US2020167608A1,Generative Adversarial Network Medical Image G...,"1 . A method, in a data processing system comp...",method data proces